In [1]:
import argparse
import time
import numpy as np
from vllm import LLM, SamplingParams

In [2]:
def generate_workloads(batch_size):
    """
    Simulates fetching prompts from your MMLU dataset.
    """
    # 1. Pathological Workload: Forces routing to a very narrow subset of experts.
    pathological_prompts = [
        "Please explain the fundamental theorems of Abstract Algebra in deep mathematical detail. " * 5
    ] * batch_size

    # 2. Ideal Workload: Requests evenly distributed across many domains.
    domains = [
        "Explain Abstract Algebra concepts. " * 5,
        "Detail the history of the Roman Empire. " * 5,
        "Describe the cellular respiration process. " * 5,
        "Analyze the legal precedent of contract law. " * 5,
        "Summarize the plot of Shakespeare's Hamlet. " * 5,
        "Explain the principles of quantum mechanics. " * 5,
        "Discuss the macroeconomic impacts of inflation. " * 5,
        "Write a Python script for data analysis. " * 5,
    ]
    
    ideal_prompts = []
    for i in range(batch_size):
        ideal_prompts.append(domains[i % len(domains)])

    return pathological_prompts, ideal_prompts

In [3]:
def run_workload(llm, prompts, sampling_params, workload_name):
    
    print(f"\n--- Running {workload_name} Workload ---")
    
    print("Performing warmup run...")
    n_warmup_prompts = 4
    _ = llm.generate(prompts[:n_warmup_prompts], sampling_params, use_tqdm=False)
    
    print(f"Executing batch of {len(prompts)} requests...")
    
    start_time = time.perf_counter()
    outputs = llm.generate(prompts, sampling_params, use_tqdm=True)
    end_time = time.perf_counter()

    tpots = []
    for output in outputs:
        metrics = output.metrics
        if metrics.first_token_time and metrics.finished_time:
            generation_time = metrics.finished_time - metrics.first_token_time
            num_tokens = len(output.outputs[0].token_ids)
            
            if num_tokens > 1:
                tpot = (generation_time / (num_tokens - 1)) * 1000 
                tpots.append(tpot)

    avg_tpot = np.mean(tpots)
    p99_tpot = np.percentile(tpots, 99)
    total_time = end_time - start_time

    print(f"Results for {workload_name}:")
    print(f"  Total Batch Time: {total_time:.2f} seconds")
    print(f"  Average TPOT:     {avg_tpot:.2f} ms/token")
    print(f"  P99 TPOT:         {p99_tpot:.2f} ms/token")
    
    return avg_tpot

In [ ]:
#model = "Qwen/Qwen1.5-MoE-A2.7B-Chat"
model = "deepseek-ai/DeepSeek-V2-Lite-Chat"
#model = 'mistralai/Mixtral-8x7B-Instruct-v0.1'
n_gpus = 1
batch_size = 1

llm = LLM(
        model=model,
        tensor_parallel_size=n_gpus,
        data_parallel_size=1,        # EP size = DP size * TP size
        enable_expert_parallel=True, # Enable EP
        quantization='bitsandbytes',          # Enable quantization
        trust_remote_code=True,
        enforce_eager=False 
    )

sampling_params = SamplingParams(
        temperature=0.0, 
        min_tokens=0,
        max_tokens=100,
        ignore_eos=True  
    )

INFO 04-15 13:28:18 [utils.py:233] non-default args: {'trust_remote_code': True, 'enable_expert_parallel': True, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'deepseek-ai/DeepSeek-V2-Lite-Chat'}


configuration_deepseek.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/deepseek-ai/DeepSeek-V2-Lite-Chat:
- configuration_deepseek.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


INFO 04-15 13:28:19 [config.py:446] Replacing legacy 'type' key with 'rope_type'
INFO 04-15 13:28:29 [model.py:549] Resolved architecture: DeepseekV2ForCausalLM
WARNING 04-15 13:28:29 [model.py:1963] Your device 'Quadro RTX 5000' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 04-15 13:28:29 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 04-15 13:28:29 [model.py:1678] Using max model len 163840
INFO 04-15 13:28:29 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-15 13:28:29 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=771843) INFO 04-15 13:28:30 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='deepseek-ai/DeepSeek-V2-Lite-Chat', speculative_config=None, tokenizer='deepseek-ai/DeepSeek-V2-Lite-Chat', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=tor

(EngineCore pid=771843) ERROR 04-15 13:28:33 [fa_utils.py:145] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(EngineCore pid=771843) ERROR 04-15 13:28:33 [fa_utils.py:145] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(EngineCore pid=771843) ERROR 04-15 13:28:33 [fa_utils.py:145] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(EngineCore pid=771843) ERROR 04-15 13:28:33 [fa_utils.py:145] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(EngineCore pid=771843) ERROR 04-15 13:28:33 [fa_utils.py:145] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(EngineCore pid=771843) ERROR 04-15 13:28:33 [fa_utils.py:145] Cannot use FA version 2 is not supported due to FA2 is only supported on

...

(EngineCore pid=771843) INFO 04-15 13:29:27 [gpu_model_runner.py:4820] Model loading took 9.08 GiB memory and 55.297877 seconds
(EngineCore pid=771843) INFO 04-15 13:29:35 [backends.py:1051] Using cache directory: /home/dylan/.cache/vllm/torch_compile_cache/df1263b27b/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=771843) INFO 04-15 13:29:35 [backends.py:1111] Dynamo bytecode transform time: 7.56 s
(EngineCore pid=771843) INFO 04-15 13:29:44 [backends.py:372] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=771843) INFO 04-15 13:29:54 [backends.py:390] Compiling a graph for compile range (1, 8192) takes 17.91 s
(EngineCore pid=771843) INFO 04-15 13:29:56 [decorators.py:640] saved AOT compiled function to /home/dylan/.cache/vllm/torch_compile_cache/torch_aot_compile/4c691a7faa9ad9f80c67687ca3221e6052fade219263f145d17d87e7965d7cea/rank_0_0/model
(EngineCore pid=771843) INFO 04-15 13:29:56 [monitor.py:48] torch.compile took 28.58 s in total


(EngineCore pid=771843) /home/dylan/.local/lib/python3.10/site-packages/torch/_inductor/lowering.py:7627: UserWarning: 
(EngineCore pid=771843) Online softmax is disabled on the fly since Inductor decides to
(EngineCore pid=771843) split the reduction. Cut an issue to PyTorch if this is an
(EngineCore pid=771843) important use case and you want to speed it up with online
(EngineCore pid=771843) softmax.
(EngineCore pid=771843) 
(EngineCore pid=771843)   warnings.warn(
(EngineCore pid=771843) /home/dylan/.local/lib/python3.10/site-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Quadro RTX 5000 does not support bfloat16 compilation natively, skipping
(EngineCore pid=771843)   warnings.warn(


(EngineCore pid=771843) WARNING 04-15 13:29:59 [fused_moe.py:1090] Using default MoE config. Performance might be sub-optimal! Config file not found at /home/dylan/.local/lib/python3.10/site-packages/vllm/model_executor/layers/fused_moe/configs/E=64,N=1408,device_name=Quadro_RTX_5000.json
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108] EngineCore failed to start.
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108] Traceback (most recent call last):
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]   File "/home/dylan/.local/lib/python3.10/site-packages/triton/language/core.py", line 43, in wrapper
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]     return fn(*args, **kwargs)
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]   File "/home/dylan/.local/lib/python3.10/site-packages/triton/language/core.py", line 2054, in dot
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]     res = _semantic.dot(input, other, acc, input_preci

(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]   File "/home/dylan/.local/lib/python3.10/site-packages/torch/fx/graph_module.py", line 455, in __call__
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]     raise e
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]   File "/home/dylan/.local/lib/python3.10/site-packages/torch/fx/graph_module.py", line 442, in __call__
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]     return super(self.cls, obj).__call__(*args, **kwargs)  # type: ignore[misc]
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]   File "/home/dylan/.local/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]     return self._call_impl(*args, **kwargs)
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]   File "/home/dylan/.local/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1787, in _call_impl
(Engin

(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]     result = self.quant_method.apply(
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/model_executor/layers/quantization/bitsandbytes.py", line 494, in apply
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]     return fused_experts(
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/model_executor/layers/fused_moe/fused_moe.py", line 1575, in fused_experts
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]     return dispatch_fused_experts_func(inplace)(
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]   File "/home/dylan/.local/lib/python3.10/site-packages/vllm/model_executor/layers/fused_moe/fused_moe.py", line 1545, in torch_vllm_outplace_fused_experts
(EngineCore pid=771843) ERROR 04-15 13:29:59 [core.py:1108]     return torch.ops.vllm.outplace_fused_e

In [ ]:
pathological_prompts, ideal_prompts = generate_workloads(batch_size)

In [ ]:
ideal_tpot = run_workload(llm, ideal_prompts, sampling_params, "Ideal (Mixed Domains)")
pathological_tpot = run_workload(llm, pathological_prompts, sampling_params, "Pathological (Single Domain)")

In [ ]:
degradation = ((pathological_tpot - ideal_tpot) / ideal_tpot) * 100
    print(f"\n======================================")
    print(f"EXPERIMENT CONCLUSION")
    print(f"TPOT Degradation due to Load Imbalance: +{degradation:.2f}%")
    print(f"======================================")